In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

os.makedirs("../results/figures", exist_ok=True)
pd.set_option("display.float_format", "{:.3f}".format)

print("Librairies chargées ✓")

Librairies chargées ✓


In [8]:
prices  = pd.read_csv("../data/prices.csv",
                       index_col=0, parse_dates=True)
dataset = pd.read_csv("../data/dataset.csv",
                       parse_dates=["date"])

print(f"Prix     : {prices.shape}")
print(f"Dataset  : {dataset.shape}")
print(f"\nColonnes du dataset :")
print(list(dataset.columns))

Prix     : (1249, 22)
Dataset  : (24533, 16)

Colonnes du dataset :
['ticker', 'date', 'label', 'trailingPE', 'priceToBook', 'debtToEquity', 'returnOnEquity', 'returnOnAssets', 'profitMargins', 'currentRatio', 'floatShares', 'momentum_1m', 'momentum_3m', 'momentum_6m', 'volatility', 'rel_strength']


## Création du label de surperformance

Le label est la variable qu'on cherche à prédire. Pour chaque action et chaque date, on regarde ce qui se passe **252 jours plus tard** (1 an de trading) :

- **Label 1** → l'action a fait mieux que l'indice JSE All Share
- **Label 0** → l'action a fait moins bien que l'indice

C'est ce qu'on appelle un problème de **classification binaire**.

In [9]:
# Taux de surperformance par ticker
label_by_ticker = dataset.groupby("ticker")["label"].mean() * 100
label_by_ticker = label_by_ticker.sort_values(ascending=True).reset_index()
label_by_ticker.columns = ["ticker", "taux_surperf"]

fig = px.bar(
    label_by_ticker,
    x="taux_surperf",
    y="ticker",
    orientation="h",
    title="Taux de surperformance par action (2019-2024)",
    labels={
        "taux_surperf": "Surperformance (%)",
        "ticker": "Action"
    },
    color="taux_surperf",
    color_continuous_scale=["#D85A30", "#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=700, height=600
)

fig.add_vline(
    x=50, line_dash="dash",
    line_color="gray",
    annotation_text="50% (ligne de référence)"
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/02_label_par_ticker.html")
fig.write_image("../results/figures/02_label_par_ticker.png")

Resorting to unclean kill browser.


##  Interprétation

Aucune action ne dépasse la ligne des 50% — ce qui signifie que sur la période 2019-2024, **aucune action JSE de notre univers n'a surperformé l'indice plus de la moitié du temps**.C'est un résultat important : battre le marché est difficile, même pour les grandes entreprises sud-africaines.

**GFI.JO (Gold Fields)** et **WHL.JO (Woolworths)** sont les plus proches de la ligne avec ~50% — ce sont les actions les plus régulièrement compétitives face à l'indice.

**SPP.JO (Shoprite)** est en queue de classement avec moins de 5% de surperformance — surprenant pour un distributeur dominant, mais qui s'explique par la forte pondération de Shoprite dans l'indice JSE lui-même, ce qui rend la surperformance mécaniquement difficile.

Ce déséquilibre confirme le **taux global de 33%** qu'on avait observé dans le dataset — le label 0 domine largement.

In [10]:
# Est-ce que certaines années sont meilleures que d'autres ?
dataset["year"] = dataset["date"].dt.year
label_by_year   = dataset.groupby("year")["label"].mean() * 100

fig = px.bar(
    x=label_by_year.index,
    y=label_by_year.values,
    title="Taux de surperformance moyen par année",
    labels={"x": "Année", "y": "Taux de surperformance (%)"},
    color=label_by_year.values,
    color_continuous_scale=["#D85A30", "#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=700, height=400
)

fig.add_hline(
    y=50, line_dash="dash",
    line_color="gray",
    annotation_text="50%"
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/02_label_par_annee.html")
fig.write_image("../results/figures/02_label_par_annee.png")

Resorting to unclean kill browser.


##  Interprétation

Le graphique révèle un pattern très clair en trois phases.

**2019** démarre faiblement à ~30% — peu d'actions battent l'indice cette année là, le marché JSE était globalement porté par quelques grands titres.

**2020 et 2021** sont les meilleures années avec ~50% et légèrement au dessus — la période post-Covid a créé une forte reprise généralisée où beaucoup d'actions ont profité du rebond, facilitant la surperformance individuelle.

**2022** retombe à ~33% et **2023 est à 0%**. Le 0% de 2023 s'explique techniquement : notre label est calculé sur 252 jours dans le futur, donc les données de 2023 regardent vers fin 2024 qui dépasse notre dataset. Ces lignes ont toutes le label 0 par construction — c'est la limite qu'on avait identifiée lors du preprocessing.

C'est pourquoi on avait choisi **2022 comme année de test** pour le modèle ML — c'est la dernière année avec des labels fiables et équilibrés.

In [11]:
# Visualisation des nouvelles features qu'on a ajoutées
dynamic_features = ["momentum_1m", "momentum_3m",
                     "momentum_6m", "volatility", "rel_strength"]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=dynamic_features + [""]
)

positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]
colors    = ["#1D9E75", "#534AB7", "#D85A30", "#BA7517", "#185FA5"]

for feat, (row, col), color in zip(dynamic_features, positions, colors):
    fig.add_trace(
        go.Histogram(
            x=dataset[feat],
            name=feat,
            showlegend=False,
            marker_color=color,
            opacity=0.8,
            nbinsx=50
        ),
        row=row, col=col
    )

fig.update_layout(
    title="Distribution des features dynamiques",
    template="plotly_white",
    height=550, width=900
)

fig.show()

fig.write_html("../results/figures/02_features_dynamiques.html")
fig.write_image("../results/figures/02_features_dynamiques.png")

c:\Users\HP\Desktop\Data Portfolio\Africa_ml_stock\venv\Lib\site-packages\choreographer\utils\_tmpfile.py:137: TmpDirWarning: The temporary directory could not be deleted, execution will continue. errors: [(WindowsPath('C:/Users/HP/AppData/Local/Temp/tmpeesvmwdx/CrashpadMetrics-active.pma'), PermissionError(13, 'Accès refusé')), (WindowsPath('C:/Users/HP/AppData/Local/Temp/tmpeesvmwdx'), OSError(41, 'Le répertoire n’est pas vide'))]
  warnings.warn(  # noqa: B028
Temporary dictory couldn't be removed manually.


## Interprétation

Toutes les distributions sont **fortement concentrées autour de zéro** avec de longues queues à droite — c'est ce qu'on appelle une distribution asymétrique à droite (skewed right).

Les **momentum_1m, 3m et 6m** sont centrés sur 0 avec la grande majorité des valeurs entre -0.2 et +0.5. La queue droite s'étend jusqu'à 2 voire 4 — ce sont des hausses exceptionnelles comme le rebond post-Covid de certaines actions en 2020-2021.

La **volatilité** a une distribution différente — elle ne peut pas être négative. Elle est concentrée entre 0.01 et 0.05 (1% à 5% de variation journalière), ce qui est typique pour des actions de grandes capitalisations. Les valeurs au-delà de 0.10 correspondent aux périodes de forte turbulence.

La **force relative** ressemble aux momentum — concentrée autour de 0, ce qui signifie que la plupart du temps les actions suivent l'indice 
de près. Les valeurs extrêmes à droite représentent les rares moments où une action décroche fortement par rapport au marché.

In [12]:
# Quelle feature est la plus corrélée avec la surperformance ?
features = [
    "trailingPE", "priceToBook", "debtToEquity",
    "returnOnEquity", "returnOnAssets", "profitMargins",
    "currentRatio", "floatShares", "momentum_1m",
    "momentum_3m", "momentum_6m", "volatility", "rel_strength"
]

correlations = dataset[features + ["label"]].corr()["label"].drop("label")
correlations = correlations.sort_values()

fig = px.bar(
    x=correlations.values,
    y=correlations.index,
    orientation="h",
    title="Corrélation des features avec le label de surperformance",
    labels={"x": "Corrélation", "y": "Feature"},
    color=correlations.values,
    color_continuous_scale=["#D85A30", "white", "#1D9E75"],
    color_continuous_midpoint=0,
    template="plotly_white",
    width=700, height=500
)

fig.add_vline(x=0, line_color="gray", line_width=1)
fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/02_correlation_features.html")
fig.write_image("../results/figures/02_correlation_features.png")

c:\Users\HP\Desktop\Data Portfolio\Africa_ml_stock\venv\Lib\site-packages\choreographer\utils\_tmpfile.py:137: TmpDirWarning: The temporary directory could not be deleted, execution will continue. errors: [(WindowsPath('C:/Users/HP/AppData/Local/Temp/tmp4bc2rxs2/CrashpadMetrics-active.pma'), PermissionError(13, 'Accès refusé')), (WindowsPath('C:/Users/HP/AppData/Local/Temp/tmp4bc2rxs2'), OSError(41, 'Le répertoire n’est pas vide'))]
  warnings.warn(  # noqa: B028
Temporary dictory couldn't be removed manually.


## Interprétation

Ce graphique révèle une surprise majeure : la **volatilité** est de loin la feature la plus corrélée avec la surperformance avec un score de ~0.25 — bien au dessus de toutes les autres. Cela signifie que sur la JSE, les actions les plus volatiles ont tendance à surperformer l'indice. C'est contre-intuitif mais cohérent avec un marché émergent où le risque est davantage récompensé que sur les marchés développés.

**currentRatio, momentum_3m et profitMargins** arrivent ensuite entre 0.05 et 0.10 — une corrélation faible mais réelle. Les entreprises liquides, avec un bon momentum récent et de bonnes marges ont légèrement plus de chances de surperformer.

**debtToEquity** est la seule feature négativement corrélée à ~-0.10 — les entreprises très endettées surperforment moins souvent. C'est parfaitement cohérent avec la théorie financière : la dette est un frein à la performance.

**returnOnEquity et returnOnAssets** sont presque à zéro — surprenant, car ce sont des ratios de rentabilité classiquement importants. Cela suggère que la rentabilité passée seule ne suffit pas à prédire la surperformance future sur la JSE.

Ces corrélations faibles expliquent en grande partie pourquoi notre modèle ML obtient un AUC modeste — il n'existe pas de signal fort et simple dans ces données.